# 05 — Quality Control

**Prerequisites:** Run [03_data_acquisition.ipynb](03_data_acquisition.ipynb) first. Requires `data/norman_raw.h5ad`.

**What you'll learn:**
- Standard scRNA-seq QC metrics applied to Perturb-seq data
- Why per-perturbation QC visualization matters (perturbations confound QC metrics)
- Guide-level QC: UMI count distributions, bimodal detection threshold
- Guide assignment status per cell (0, 1, or 2+ guides)
- How to set and justify QC thresholds
- Identifying underpowered perturbations after filtering

**Estimated time:** 2 hours

## 0. Setup

In [ ]:
import os
import scanpy as sc
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.sparse

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white")

DATA_DIR = "data"
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

adata = sc.read_h5ad(os.path.join(DATA_DIR, "norman_raw.h5ad"))
print(adata)

# Identify perturbation column
PERT_COL = "perturbation" if "perturbation" in adata.obs.columns else "condition"
print(f"\nPerturbation column: '{PERT_COL}'")
print(f"Total perturbations: {adata.obs[PERT_COL].nunique()}")

## 1. Ensure we are working with raw counts

QC metrics must be computed on raw integer UMI counts, not normalized values. If `adata.X` contains normalized values, use `adata.layers['counts']` or `adata.raw`.

In [ ]:
# Check if X contains raw counts
X_sample = adata.X[:10, :10]
if scipy.sparse.issparse(X_sample):
    X_sample = X_sample.toarray()
    
is_integer = np.allclose(X_sample, np.round(X_sample))
print(f"adata.X is {'integer counts' if is_integer else 'normalized (floats)'}")

if not is_integer:
    if "counts" in adata.layers:
        print("Using adata.layers['counts'] as raw counts")
        adata_raw = adata.copy()
        adata_raw.X = adata.layers["counts"]
    elif adata.raw is not None:
        print("Using adata.raw")
        adata_raw = adata.raw.to_adata()
        # Transfer obs metadata from original
        adata_raw.obs = adata.obs.copy()
    else:
        raise ValueError("Cannot find raw counts. Check adata.layers or adata.raw.")
else:
    adata_raw = adata.copy()
    
print(f"Working with: {adata_raw.shape}")

## 2. Standard scRNA-seq QC metrics

The three standard QC metrics:
- **nGenes** (`n_genes_by_counts`): number of genes with ≥1 UMI in the cell
- **nUMI** (`total_counts`): total UMI count per cell
- **pct_counts_MT**: fraction of UMIs mapping to mitochondrial genes (high → damaged/dying cell)

In [ ]:
# Mark mitochondrial genes (human: prefix 'MT-')
adata_raw.var["mt"] = adata_raw.var_names.str.startswith("MT-")
mt_count = adata_raw.var["mt"].sum()
print(f"Mitochondrial genes detected: {mt_count}")

sc.pp.calculate_qc_metrics(
    adata_raw,
    qc_vars=["mt"],
    percent_top=None,
    log1p=False,
    inplace=True,
)
print("QC metrics computed.")
print(adata_raw.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe())

In [ ]:
# Overall QC violin plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric, label in zip(
    axes,
    ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
    ["Total UMIs", "Genes detected", "% MT reads"],
):
    ax.violinplot(adata_raw.obs[metric], positions=[0], showmedians=True)
    ax.set_ylabel(label)
    ax.set_xticks([])
    ax.set_title(label)

plt.suptitle("Norman 2019 — Overall QC", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "05_qc_overall.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. Per-perturbation QC — a critical Perturb-seq step

**Do not filter cells based on global QC thresholds alone.** Some perturbations genuinely cause lower nUMI (e.g., growth-arrest perturbations → smaller cells → fewer RNA molecules) or higher pct_counts_MT (e.g., mitochondrial pathway perturbations). Applying a global nUMI threshold would selectively remove cells from these perturbations, introducing systematic bias.

Always check QC distributions per perturbation before setting thresholds.

In [ ]:
# Identify NTC cells
ntc_labels = [x for x in adata_raw.obs[PERT_COL].unique()
              if any(kw in str(x).lower() for kw in ["ctrl", "control", "non", "ntc"])]
print(f"NTC labels: {ntc_labels}")

# Per-perturbation median UMI
pert_qc = adata_raw.obs.groupby(PERT_COL)[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].median()
pert_qc = pert_qc.sort_values("total_counts")

fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
for ax, col, label in zip(
    axes,
    ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
    ["Median UMIs", "Median genes", "Median %MT"],
):
    colors = ["red" if p in ntc_labels else "steelblue" for p in pert_qc.index]
    ax.bar(range(len(pert_qc)), pert_qc[col], color=colors, edgecolor="none", width=1)
    ax.set_ylabel(label)
    ax.tick_params(axis="x", bottom=False, labelbottom=False)

axes[-1].set_xlabel("Perturbation (sorted by median UMI)")
plt.suptitle("Per-perturbation QC medians (red = NTC)", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "05_qc_per_perturbation.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Perturbations with lowest median UMI:")
print(pert_qc["total_counts"].head(10))

## 4. Guide UMI distribution

The Norman 2019 labeled h5ad stores the final perturbation assignment in `adata.obs`, not a raw guide UMI count matrix. In a freshly processed dataset (from Cell Ranger), you would have a guide × cell UMI matrix.

Here we demonstrate what that distribution looks like conceptually, and then work with the assignment metadata already in `obs`.

Key concepts:
- Cells with a guide show a **bimodal** distribution of guide UMIs: a low peak (background/noise) and a high peak (true capture)
- Assignment threshold: cells with top-guide UMI count above the valley between the two peaks are considered assigned
- Guide purity score = (top guide UMI) / (total guide UMI) — cells with purity < 0.75 may be multiplets

In [ ]:
# In the Norman labeled h5ad, guide information is in obs columns
# Typically: 'guide_ids' (which guides detected), 'UMI_count' (guide UMI per cell)
guide_related_cols = [c for c in adata_raw.obs.columns if any(
    kw in c.lower() for kw in ["guide", "umi", "gRNA", "barcode", "n_guides"]
)]
print("Guide-related obs columns:", guide_related_cols)

# Show a sample
if guide_related_cols:
    print(adata_raw.obs[guide_related_cols].head(10))

In [ ]:
# Simulate guide UMI distribution for pedagogical purposes
# (In a real dataset from Cell Ranger, you would load the guide capture matrix directly)
np.random.seed(42)
n_cells = 5000

# Simulate: 80% transduced cells (1 guide), 5% multiplets, 15% empty/low
guide_umis = np.concatenate([
    np.random.poisson(1.5, int(n_cells * 0.15)),   # background
    np.random.poisson(35, int(n_cells * 0.80)),     # true capture
    np.random.poisson(35, int(n_cells * 0.05)) +    # multiplets: two guides
    np.random.poisson(20, int(n_cells * 0.05)),
])

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(guide_umis, bins=60, color="steelblue", edgecolor="none", alpha=0.8)
ax.axvline(8, color="red", linestyle="--", linewidth=2, label="Assignment threshold")
ax.set_xlabel("Guide capture UMI count (top guide per cell)")
ax.set_ylabel("Number of cells")
ax.set_title("Simulated guide UMI distribution (bimodal)")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "05_guide_umi_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

print("The left peak: cells with low guide UMI (failed transduction or empty droplets)")
print("The right peak: cells with a true guide capture event")
print("The threshold (red line) separates assigned from unassigned cells.")

## 5. Guide assignment status per cell

In [ ]:
# In the Norman labeled dataset, perturbation assignment is already done.
# We classify cells into:
#   - NTC: non-targeting control
#   - single: one gene targeted
#   - dual: two genes targeted (contains '+')

def classify_cell(pert):
    p = str(pert)
    if any(kw in p.lower() for kw in ["ctrl", "control", "non"]):
        return "NTC"
    elif "+" in p:
        return "dual"
    else:
        return "single"

adata_raw.obs["assignment_type"] = adata_raw.obs[PERT_COL].apply(classify_cell)

counts = adata_raw.obs["assignment_type"].value_counts()
print("Assignment type counts:")
print(counts)

fig, ax = plt.subplots(figsize=(5, 4))
counts.plot(kind="bar", ax=ax, color=["#2196F3", "#4CAF50", "#FF9800"], edgecolor="none")
ax.set_xlabel("Assignment type")
ax.set_ylabel("Cell count")
ax.set_title("Guide assignment status")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "05_assignment_status.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Set and apply QC thresholds

Recommended thresholds for Perturb-seq (adjust based on your distribution):

| Metric | Remove if | Rationale |
|---|---|---|
| `total_counts` | < 500 or > 99th percentile | Empty droplets or doublets |
| `n_genes_by_counts` | < 200 | Low-quality cells |
| `pct_counts_mt` | > 20% | Dying/apoptotic cells |

These are starting points — **always look at the per-perturbation distributions** (section 3) before deciding whether an outlier perturbation should be excluded from filtering.

In [ ]:
# Compute thresholds from the data
total_counts_low = 500
total_counts_high = np.percentile(adata_raw.obs["total_counts"], 99)
n_genes_low = 200
pct_mt_high = 20.0

print(f"Thresholds:")
print(f"  total_counts: [{total_counts_low}, {total_counts_high:.0f}]")
print(f"  n_genes_by_counts: >= {n_genes_low}")
print(f"  pct_counts_mt: <= {pct_mt_high}%")

keep = (
    (adata_raw.obs["total_counts"] >= total_counts_low) &
    (adata_raw.obs["total_counts"] <= total_counts_high) &
    (adata_raw.obs["n_genes_by_counts"] >= n_genes_low) &
    (adata_raw.obs["pct_counts_mt"] <= pct_mt_high)
)

print(f"\nCells before filtering: {len(adata_raw)}")
print(f"Cells removed: {(~keep).sum()} ({(~keep).mean()*100:.1f}%)")
print(f"Cells after filtering: {keep.sum()}")

In [ ]:
# Check how many cells are removed per perturbation
removal_rate = adata_raw.obs.copy()
removal_rate["removed"] = ~keep
per_pert_removal = removal_rate.groupby(PERT_COL)["removed"].mean().sort_values(ascending=False)

print("Top 10 perturbations by removal rate:")
print(per_pert_removal.head(10))

high_removal = per_pert_removal[per_pert_removal > 0.3]
if len(high_removal) > 0:
    print(f"\nWarning: {len(high_removal)} perturbation(s) lose >30% of cells — check if biologically expected:")
    print(high_removal)

In [ ]:
# Apply filters
adata_qc = adata_raw[keep].copy()
print(f"After QC filtering: {adata_qc.shape}")

## 7. Post-filter: check cells per perturbation

After filtering, flag any perturbations with fewer than 20 cells — these are statistically underpowered for DE analysis.

In [ ]:
cells_post_qc = adata_qc.obs[PERT_COL].value_counts()

underpowered = cells_post_qc[cells_post_qc < 20]
print(f"Perturbations with < 20 cells after QC: {len(underpowered)}")
if len(underpowered) > 0:
    print(underpowered)

# Annotate underpowered perturbations
adata_qc.obs["is_underpowered"] = adata_qc.obs[PERT_COL].isin(underpowered.index)

fig, ax = plt.subplots(figsize=(16, 4))
colors = ["red" if c < 20 else "steelblue" for c in cells_post_qc.values]
cells_post_qc.plot(kind="bar", ax=ax, color=colors, edgecolor="none")
ax.axhline(20, color="red", linestyle="--", linewidth=1.5, label="20-cell minimum")
ax.set_xlabel("Perturbation")
ax.set_ylabel("Cells")
ax.set_title("Cells per perturbation after QC filtering")
ax.tick_params(axis="x", rotation=90, labelsize=6)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "05_cells_per_pert_post_qc.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Save QC-filtered AnnData

In [ ]:
out_path = os.path.join(DATA_DIR, "norman_qc.h5ad")
adata_qc.write_h5ad(out_path)
print(f"Saved QC-filtered AnnData to {out_path}")
print(f"Shape: {adata_qc.shape}")

## Key takeaways

1. Apply standard scRNA-seq QC (nUMI, nGenes, %MT) but **always inspect distributions per perturbation** — some perturbations will genuinely produce low-UMI or high-MT cells
2. The guide UMI distribution is bimodal — the valley between peaks determines the assignment threshold
3. Guide purity score (top guide UMI / total guide UMI) identifies multiplets
4. After filtering, flag perturbations with < 20 cells as statistically underpowered
5. Record and justify every threshold — QC is a judgment call, not an algorithm

**Next:** [06_guide_assignment.ipynb](06_guide_assignment.ipynb)